# AGDM

Trying to get AGSL's digital map collection indexed into GeoDiscovery.
Crosswalk metadata from CONTENTdm to OpenGeoMetadata Aardvark.
Evaluate the suitability of the RDA bounding boxes populated from catalog records.

Furthermore, it would be great to develop a workflow to extract meaningful bounding box/spatial index inforamation from maps georeferenced with Allmaps.

In [3]:
import csv
from itertools import islice

with open(r'agdm_data/agdm-Pilot.csv', 'r', newline='', encoding='utf-8') as csv_file:
    r = csv.DictReader(csv_file)
    for row in islice(r, 5):
        print(f'{row["Short Title"]}: \t {row["Bounding Box"]}')

Holy Land Maps #93: 	 E 30°--E 38°/N 34°--N 29°
Holy Land Maps #97: 	 E 33°15ʹ--E 38°40ʹ/N 35°00ʹ--N 29°00ʹ
Argentina 1995: 	 W 81⁰--W 50⁰/S 21⁰--S 57⁰
Arizona 1963: 	 W 114°49ʹ00ʺ--W 109°02ʹ00ʺ/N 37°00ʹ00ʺ--N 31°19ʹ00ʺ
Mumbai, India 1947: 	 E 72°47ʹ07ʺ--E 72°53ʹ13ʺ/N 19°03ʹ34ʺ--N 18°53ʹ17ʺ


In [11]:
import json
from pathlib import Path

data_path = Path('agdm_data/annotations')
assert data_path.exists() and data_path.is_dir(), f"Data path {data_path} does not exist or is not a directory"

In [23]:
for file in data_path.glob('*.json'):
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        print(f'File: {file.name}')

File: 29d7e689c8ee71ed.json
File: a5a82fdc52322ee6.json
File: bfd6c8e5fad5cd75.json
File: 3795d36d0f7fd4c1.json
File: 90c6bb18c7f47737.json
File: 21dc89c8be7dfd38.json
File: 6d34f9569f56d613.json
File: ae96df6c435b7c2b.json
File: 0f8dee5242b7fa45.json
File: b7f1649427ae47f6.json
File: 8831d4255fa7a934.json
File: 5802f61196218b6e.json
File: 520dabbf896af46f.json
File: 9265afe0fb58d6a9.json


In [24]:
def to_cdm_link(data):
    """
    Convert AGDM data to CDM link format.
    """
    provider = data["target"]["source"]["provider"][0]

    cdm_link = {
        "site": provider["homepage"][0]["id"],
    }

    return cdm_link

In [27]:
import xml.etree.ElementTree as ET

def extract_mask_points(data):
    svg = data["target"]["selector"]["value"]
    root = ET.fromstring(svg)

    polygon = root.find('.//polygon')
    points = polygon.attrib['points']

    return [
        tuple(map(float, point.split(','))) 
        for point in points.strip().split(' ')
    ]


In [28]:
for file in data_path.glob('*.json'):
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        mask_points = extract_mask_points(data)
        print(mask_points.__str__())

[(520.0, 10121.0), (7646.0, 10107.0), (7653.0, 1106.0), (526.0, 1087.0)]
[(291.0, 592.0), (277.0, 11238.0), (11690.0, 11249.0), (11649.0, 599.0)]
[(615.0, 737.0), (587.0, 10888.0), (11596.0, 10881.0), (11616.0, 736.0)]
[(370.0, 260.0), (16870.0, 270.0), (16870.0, 12670.0), (375.0, 12670.0)]
[(563.0, 720.0), (599.0, 9876.0), (13043.0, 9855.0), (13036.0, 756.0)]
[(486.0, 577.0), (457.0, 13467.0), (9998.0, 13438.0), (10099.0, 534.0)]
[(717.0, 715.0), (715.0, 4317.0), (6362.0, 4302.0), (6362.0, 707.0)]
[(400.0, 393.0), (412.0, 10198.0), (12905.0, 10184.0), (12925.0, 392.0)]
[(608.0, 918.0), (615.0, 10670.0), (13713.0, 10714.0), (13739.0, 5393.0), (11086.0, 5371.0), (9633.0, 4196.0), (9639.0, 906.0)]
[(9773.0, 605.0), (9766.0, 4130.0), (11129.0, 5237.0), (13744.0, 5251.0), (13779.0, 674.0)]
[(399.0, 422.0), (453.0, 10998.0), (12266.0, 11015.0), (12250.0, 413.0)]
[(458.0, 827.0), (480.0, 15692.0), (7989.0, 15706.0), (8019.0, 817.0)]
[(585.0, 755.0), (558.0, 10261.0), (6609.0, 10274.0), (1273

In [43]:
import subprocess

from shapely.geometry import shape, mapping


def allmaps_mask_to_geom(annotation_path) -> shape:
    result = subprocess.run(
        [
            "allmaps",
            "transform",
            "resource-mask",
            str(annotation_path),
        ],
        check=True,
        capture_output=True,
        text=True,
    )

    geojson = json.loads(result.stdout)
    geom = shape(geojson["features"][0]["geometry"])

    return geom

In [53]:
annotation_path = Path("agdm_data/annotations/3795d36d0f7fd4c1.json")

geometry = allmaps_mask_to_geom(annotation_path)
print(mapping(geometry))
print(geometry.wkt)

{'type': 'Polygon', 'coordinates': (((-99.20900829215967, 19.467819648300996), (-99.20763866495399, 19.395517144274248), (-99.10482568760574, 19.394631839948126), (-99.10616307138451, 19.466876221134168), (-99.20900829215967, 19.467819648300996)),)}
POLYGON ((-99.20900829215967 19.467819648300996, -99.20763866495399 19.395517144274248, -99.10482568760574 19.394631839948126, -99.10616307138451 19.466876221134168, -99.20900829215967 19.467819648300996))
